In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Helper to convert timestamp column to timezone-aware datetime in UTC+8 and attach as 'ts'
def attach_ts(df):
    if 'timestamp' in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df['ts'] = pd.NaT
    return df

metric_df = attach_ts(metric_df)
trace_df = attach_ts(trace_df)
log_df = attach_ts(log_df)
error_df = attach_ts(error_df)

# Summaries function
def file_overview(df, fname):
    rows = df.shape[0]
    min_ts = df['ts'].min() if rows>0 else pd.NaT
    max_ts = df['ts'].max() if rows>0 else pd.NaT
    unique_cmdb = df['cmdb_id'].nunique() if 'cmdb_id' in df.columns else 0
    return pd.DataFrame([{
        'file': fname,
        'rows': rows,
        'min_ts': min_ts,
        'max_ts': max_ts,
        'unique_cmdb_count': unique_cmdb
    }])

# Build overviews
metric_overview = file_overview(metric_df, 'metric.csv')
trace_overview = file_overview(trace_df, 'trace.csv')
log_overview = file_overview(log_df, 'log.csv')
error_overview = file_overview(error_df, 'error_logs.csv')

# Top cmdb_id counts (top 20) for each file
metric_cmdb_top = metric_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(20)
trace_cmdb_top = trace_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(20)
log_cmdb_top = log_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(20)
error_cmdb_top = error_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(20)

# Top KPI/feature/log/message values (top 20)
metric_kpi_top = metric_df['kpi_name'].value_counts().rename_axis('kpi_name').reset_index(name='count').head(20)
trace_feature_top = trace_df['trace_name'].value_counts().rename_axis('trace_name').reset_index(name='count').head(20)
log_feature_top = log_df['log_name'].value_counts().rename_axis('log_name').reset_index(name='count').head(20)
error_message_top = error_df['message'].value_counts().rename_axis('message').reset_index(name='count').head(20)

# Up to 5 sample error log rows (compact)
error_samples = error_df[['ts','cmdb_id','message']].head(5)

# Final compact outputs (display variables; IPython will show them)
metric_overview, metric_cmdb_top, metric_kpi_top, trace_overview, trace_cmdb_top, trace_feature_top, log_overview, log_cmdb_top, log_feature_top, error_overview, error_cmdb_top, error_message_top, error_samples

```
Out[1]:
```
# Summarize the earlier execution results in plain English using existing variables
def fmt_top(df, key_col, val_col='count', n=10):
    if df is None or df.shape[0]==0:
        return "None"
    items = []
    for r in df.head(n).itertuples(index=False):
        items.append(f"{getattr(r, key_col)}:{getattr(r, val_col)}")
    return ", ".join(items)

metric_rows = int(metric_overview.loc[0,'rows'])
metric_min = metric_overview.loc[0,'min_ts']
metric_max = metric_overview.loc[0,'max_ts']
metric_unique_cmdb = int(metric_overview.loc[0,'unique_cmdb_count'])
metric_top_cmdb = fmt_top(metric_cmdb_top, 'cmdb_id')
metric_top_kpis = fmt_top(metric_kpi_top, 'kpi_name')

trace_rows = int(trace_overview.loc[0,'rows'])
trace_min = trace_overview.loc[0,'min_ts']
trace_max = trace_overview.loc[0,'max_ts']
trace_unique_cmdb = int(trace_overview.loc[0,'unique_cmdb_count'])
trace_top_cmdb = fmt_top(trace_cmdb_top, 'cmdb_id')
trace_top_features = fmt_top(trace_feature_top, 'trace_name')

log_rows = int(log_overview.loc[0,'rows'])
log_min = log_overview.loc[0,'min_ts']
log_max = log_overview.loc[0,'max_ts']
log_unique_cmdb = int(log_overview.loc[0,'unique_cmdb_count'])
log_top_cmdb = fmt_top(log_cmdb_top, 'cmdb_id')
log_top_names = fmt_top(log_feature_top, 'log_name')

error_rows = int(error_overview.loc[0,'rows'])
# error top messages and samples may be empty
error_top_msgs = fmt_top(error_message_top, 'message') if 'error_message_top' in globals() else "None"
error_samples_exist = (not error_samples.empty) if 'error_samples' in globals() else False

summary = (
    "Summary of loaded telemetry files (timestamps shown in UTC+8):\n\n"
    "metric.csv:\n"
    f" - rows: {metric_rows}\n"
    f" - time range: {metric_min} to {metric_max}\n"
    f" - unique cmdb_id count: {metric_unique_cmdb}\n"
    f" - top cmdb_id by count (top 10): {metric_top_cmdb}\n"
    f" - top kpi_name values (top 10): {metric_top_kpis}\n\n"
    "trace.csv:\n"
    f" - rows: {trace_rows}\n"
    f" - time range: {trace_min} to {trace_max}\n"
    f" - unique cmdb_id count: {trace_unique_cmdb}\n"
    f" - top cmdb_id by count (top 10): {trace_top_cmdb}\n"
    f" - top trace_name values (top 10): {trace_top_features}\n\n"
    "log.csv:\n"
    f" - rows: {log_rows}\n"
    f" - time range: {log_min} to {log_max}\n"
    f" - unique cmdb_id count: {log_unique_cmdb}\n"
    f" - top cmdb_id by count (top 10): {log_top_cmdb}\n"
    f" - top log_name values (top 10): {log_top_names}\n\n"
    "error_logs.csv:\n"
    f" - rows: {error_rows}\n"
    f" - time range: None (file is empty)\n"
    f" - top messages (top 10): {error_top_msgs}\n"
    f" - sample rows present: {error_samples_exist}\n\n"
    "Notes:\n"
    " - metric.csv is the largest file (28,823 rows) covering many KPI types; top cmdb_ids include Mysql01 and Mysql02.\n"
    " - trace.csv has 6,774 rows; MG01 and MG02 are the most frequent cmdb_ids there.\n"
    " - log.csv is small (375 rows) with Tomcat nodes prominent.\n"
    " - error_logs.csv contains no rows in this dataset.\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file   rows                    min_ts                    max_ts  unique_cmdb_count
0  metric.csv  28823 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                 26,          cmdb_id  count
0        Mysql01   4592
1        Mysql02   4583
2       Tomcat04   2014
3       Tomcat03   1998
4       Tomcat02   1955
5       Tomcat01   1918
6        Redis02   1808
7           MG02   1571
8           MG01   1565
9        Redis01   1543
10          IG01   1059
11          IG02   1025
12      apache02    881
13      apache01    868
14  ServiceTest6    124
15  ServiceTest7    124
16  ServiceTest2    124
17  ServiceTest4    124
18  ServiceTest9    124
19  ServiceTest8    124,                                              kpi_name  count
0        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    345
1          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    341
2       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    341
3      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    341
4                  OSLinux-OSLinux_ZABBIX_Host_Uptime    340
5                         OSLinux-CPU_CPU_CPUidleutil    338
6   OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    338
7                         OSLinux-CPU_CPU_CPUUserTime    338
8                          OSLinux-CPU_CPU_CPUSysTime    338
9                              OSLinux-CPU_CPU_CPUWio    338
10                            OSLinux-CPU_CPU_CPULoad    338
11                                             app.rr    333
12                                            app.mrt    333
13                                            app.cnt    333
14                                             app.sr    333
15       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    287
16                         OSLinux-CPU_CPU_CPUCpuUtil    276
17       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    274
18              OSLinux-OSLinux_MEMORY_MEMORY_UserMem    261
19          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem    257,         file  rows                    min_ts                    max_ts  unique_cmdb_count
0  trace.csv  6774 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                 12,      cmdb_id  count
0       MG01   1158
1       MG02   1143
2   Tomcat02    459
3       IG02    456
4       IG01    450
5   Tomcat01    450
6   Tomcat03    450
7   Tomcat04    450
8   dockerB2    450
9   dockerB1    450
10  dockerA2    432
11  dockerA1    426,                        trace_name  count
0        trace.self.duration_mean    364
1         trace.self.duration_p95    364
2            trace.self.row_count    364
3     trace.to_MG01.duration_mean    234
4         trace.to_MG01.row_count    234
5      trace.to_MG01.duration_p95    234
6      trace.to_MG02.duration_p95    231
7         trace.to_MG02.row_count    231
8     trace.to_MG02.duration_mean    231
9    trace.from_MG01.duration_p95    121
10      trace.from_MG01.row_count    121
11  trace.from_IG02.duration_mean    121
12   trace.from_IG02.duration_p95    121
13      trace.from_IG02.row_count    121
14  trace.from_MG01.duration_mean    121
15   trace.from_MG02.duration_p95    120
16      trace.from_MG02.row_count    120
17  trace.from_IG01.duration_mean    120
18      trace.from_IG01.row_count    120
19  trace.from_MG02.duration_mean    120,       file  rows                    min_ts                    max_ts  unique_cmdb_count
0  log.csv   375 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                  6,     cmdb_id  count
0  Tomcat04     65
1  Tomcat02     64
2  Tomcat01     62
3  Tomcat03     62
4  apache02     62
5  apache01     60,            log_name  count
0   log.error_count    183
1     log.row_count    183
2  log.gc_pause_max      9,              file  rows min_ts max_ts  unique_cmdb_count
0  error_logs.csv     0    NaT    NaT                  0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [message, count]
Index: [], Empty DataFrame
Columns: [ts, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing dataframes: metric_df, trace_df, log_df (they should already exist in the kernel).
# Ensure timezone variable exists
tz = pytz.timezone('Asia/Shanghai')

# Restrict to provided possible root-cause components
components = [
    "apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"
]

# Helper to ensure ts column exists and is timezone-aware UTC+8
def ensure_ts(df):
    if 'ts' not in df.columns or df['ts'].isnull().all():
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        # make sure tz is UTC then converted (idempotent if already correct)
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    return df

metric_df = ensure_ts(metric_df)
trace_df = ensure_ts(trace_df)
log_df = ensure_ts(log_df)

# Filter to components
metric_f = metric_df[metric_df['cmdb_id'].isin(components)].copy()
trace_f = trace_df[trace_df['cmdb_id'].isin(components)].copy()
log_f = log_df[log_df['cmdb_id'].isin(components)].copy()

# Grouping and global percentile calculation helpers
def grouped_thresholds(df, group_cols, value_col='value'):
    # Ensure numeric
    df = df.copy()
    df[value_col] = pd.to_numeric(df[value_col], errors='coerce')
    # Aggregate: count, min_ts, max_ts, p95, p5
    agg = df.groupby(group_cols).agg(
        count=(value_col, 'count'),
        min_ts=('ts', 'min'),
        max_ts=('ts', 'max'),
        p95=(value_col, lambda x: x.quantile(0.95) if x.count()>0 else pd.NA),
        p5=(value_col, lambda x: x.quantile(0.05) if x.count()>0 else pd.NA)
    ).reset_index()
    # Sort by count desc
    agg = agg.sort_values('count', ascending=False).reset_index(drop=True)
    return agg

# Compute full grouped results (saved in kernel variables)
metric_grouped_full = grouped_thresholds(metric_f, ['cmdb_id', 'kpi_name'], 'value')
trace_grouped_full = grouped_thresholds(trace_f, ['cmdb_id', 'trace_name'], 'value')
log_grouped_full = grouped_thresholds(log_f, ['cmdb_id', 'log_name'], 'value')

# Prepare compact top-20 tables for display (only requested aggregated columns)
metric_grouped_top = metric_grouped_full[['cmdb_id','kpi_name','count','min_ts','max_ts','p95','p5']].head(20)
trace_grouped_top  = trace_grouped_full[['cmdb_id','trace_name','count','min_ts','max_ts','p95','p5']].head(20)
log_grouped_top    = log_grouped_full[['cmdb_id','log_name','count','min_ts','max_ts','p95','p5']].head(20)

# Display the three compact tables (kernel will show these variables)
metric_grouped_top, trace_grouped_top, log_grouped_top

```
Out[2]:
```
Summary of global thresholds (all times in UTC+8). Full grouped results are saved in kernel variables: metric_grouped_full, trace_grouped_full, log_grouped_full.

Common notes
- Time range for the returned groups: 2021-03-25 06:30:00+08:00 through 2021-03-25 07:00:00+08:00 (most groups have 31 points; some log groups have 30 or fewer).
- Components considered: apache01, apache02, Tomcat01–04, MG01–02, IG01–02, Mysql01–02, Redis01–02.

Metric groups (top by count — representative top rows)
- MG02 / OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies — count 31; p95=1.0; p5=1.0; range 06:30–07:00.
- MG02 / OSLinux-OSLinux_PROCESS_zabbix-... — count 31; p95=0.3; p5=0.3.
- IG01 / JVM-Memory_..._HeapMemoryMax — count 31; p95≈2.040136e+09; p5≈2.040136e+09.
- IG01 / JVM-Memory_..._HeapMemoryUsed — count 31; p95≈7.369532e+08; p5≈9.81647e+07.
- Tomcat03 / Tomcat-Threads...MaxThreads — count 31; p95=500; p5=500.
(See metric_grouped_full for the complete grouped table. The compact displayed top-20 include many other OSLinux and Tomcat JVM metrics; most counts = 31.)

Trace groups (top by count — representative top rows)
- MG01 / trace.from_Tomcat02.duration_p95 — count 31; p95≈0.243975; p5≈0.027225; range 06:30–07:00.
- MG01 / trace.from_Tomcat02.row_count — count 31; p95=27.5; p5=7.5.
- IG02 / trace.self.duration_p95 — count 31; p95≈0.5451; p5≈0.0530.
- IG02 / trace.to_Tomcat02.duration_mean — count 31; p95≈0.096856; p5≈0.023711.
- IG02 / trace.self.row_count — count 31; p95=194.0; p5=103.0.
(Full grouped results in trace_grouped_full.)

Log groups (top by count — representative top rows)
- Tomcat02 / log.error_count — count 31; p95=0.0; p5=0.0 (no errors in top series).
- Tomcat02 / log.row_count — count 31; p95=49.0; p5=20.0.
- Tomcat04 / log.row_count — count 31; p95=53.5; p5=25.5.
- apache02 / log.row_count — count 31; p95=105.5; p5=51.5.
- Several Tomcat/apache log.error_count entries show p95=0 (no observed errors); GC pause metrics exist with small sample counts (2–3) and p95 ≈ 0.07–0.08s.
(Full grouped results in log_grouped_full.)

If you want, I can:
- Export a compact CSV of the top N grouped thresholds per file (kept in memory), or
- Filter the grouped results to a specific component or KPI/trace/log name and show its full threshold summary.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count                    min_ts                    max_ts           p95            p5
0       MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  1.000000e+00  1.000000e+00
1       MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  3.000000e-01  3.000000e-01
2       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  2.040136e+09  2.040136e+09
3       IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  7.369532e+08  9.816470e+07
4       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  5.846119e+07  5.846119e+07
5       MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  3.000000e+00  3.000000e+00
6       MG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  7.825816e+06  7.824188e+06
7   Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  5.000000e+02  5.000000e+02
8   Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  0.000000e+00  0.000000e+00
9   Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  0.000000e+00  0.000000e+00
10  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  0.000000e+00  0.000000e+00
11  Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  0.000000e+00  0.000000e+00
12  Tomcat04       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  0.000000e+00  0.000000e+00
13  Tomcat04                            OSLinux-CPU_CPU_CPULoad     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  1.095000e+00  1.000000e+00
14  Tomcat04      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  1.000000e+00  1.000000e+00
15  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  5.000000e+02  5.000000e+02
16      MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  0.000000e+00  0.000000e+00
17  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  1.000000e+02  1.000000e+02
18  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  1.051000e+03  1.051000e+03
19  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  5.000000e-01  0.000000e+00,      cmdb_id                         trace_name  count                    min_ts                    max_ts         p95          p5
0       MG01   trace.from_Tomcat02.duration_p95     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.243975    0.027225
1       MG01      trace.from_Tomcat02.row_count     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00   27.500000    7.500000
2       IG02            trace.self.duration_p95     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.545100    0.053000
3       IG02    trace.to_Tomcat02.duration_mean     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.096856    0.023711
4       IG02        trace.to_Tomcat02.row_count     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00   22.500000    7.500000
5       IG02               trace.self.row_count     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  194.000000  103.000000
6       IG02     trace.to_Tomcat02.duration_p95     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.535200    0.046600
7       IG02           trace.self.duration_mean     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.090118    0.027727
8       MG01  trace.from_Tomcat02.duration_mean     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.076843    0.012993
9   Tomcat02          trace.from_IG02.row_count     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00   22.500000    7.500000
10  Tomcat02            trace.self.duration_p95     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.035050    0.019800
11  Tomcat02               trace.self.row_count     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  325.500000  128.000000
12  Tomcat02        trace.to_MG01.duration_mean     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.076843    0.012993
13  Tomcat02         trace.to_MG01.duration_p95     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.243975    0.027225
14  Tomcat02       trace.from_IG02.duration_p95     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.535200    0.046600
15  Tomcat02            trace.to_MG01.row_count     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00   27.500000    7.500000
16  Tomcat02      trace.from_IG02.duration_mean     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.096856    0.023711
17  Tomcat02           trace.self.duration_mean     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.012606    0.003469
18      MG01    trace.to_dockerB2.duration_mean     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.102568    0.004967
19      MG01     trace.to_dockerB2.duration_p95     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.452000    0.013350,      cmdb_id          log_name  count                    min_ts                    max_ts       p95       p5
0   Tomcat02   log.error_count     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.0000   0.0000
1   Tomcat02     log.row_count     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00   49.0000  20.0000
2   Tomcat04     log.row_count     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00   53.5000  25.5000
3   Tomcat04   log.error_count     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.0000   0.0000
4   apache02     log.row_count     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00  105.5000  51.5000
5   apache02   log.error_count     31 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00    0.0000   0.0000
6   Tomcat01     log.row_count     30 2021-03-25 06:30:00+08:00 2021-03-25 06:59:00+08:00   47.2000  24.9000
7   Tomcat01   log.error_count     30 2021-03-25 06:30:00+08:00 2021-03-25 06:59:00+08:00    0.0000   0.0000
8   apache01     log.row_count     30 2021-03-25 06:30:00+08:00 2021-03-25 06:59:00+08:00   90.1000  52.4500
9   Tomcat03     log.row_count     30 2021-03-25 06:30:00+08:00 2021-03-25 06:59:00+08:00   50.5500  24.3500
10  apache01   log.error_count     30 2021-03-25 06:30:00+08:00 2021-03-25 06:59:00+08:00    0.0000   0.0000
11  Tomcat03   log.error_count     30 2021-03-25 06:30:00+08:00 2021-03-25 06:59:00+08:00    0.0000   0.0000
12  Tomcat04  log.gc_pause_max      3 2021-03-25 06:31:00+08:00 2021-03-25 06:51:00+08:00    0.0690   0.0510
13  Tomcat03  log.gc_pause_max      2 2021-03-25 06:40:00+08:00 2021-03-25 06:57:00+08:00    0.0795   0.0705
14  Tomcat01  log.gc_pause_max      2 2021-03-25 06:41:00+08:00 2021-03-25 06:57:00+08:00    0.0795   0.0705
15  Tomcat02  log.gc_pause_max      2 2021-03-25 06:32:00+08:00 2021-03-25 06:49:00+08:00    0.0790   0.0610)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: metric_df, metric_grouped_full
tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_df has timezone-aware 'ts'
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define incident window (inclusive) in UTC+8
window_start = pd.Timestamp('2021-03-25 06:30:00', tz=tz)
window_end   = pd.Timestamp('2021-03-25 07:00:00', tz=tz)

# Filter metric rows to the incident window
metric_window = metric_df[(metric_df['ts'] >= window_start) & (metric_df['ts'] <= window_end)].copy()

# Prepare thresholds lookup (from precomputed global thresholds)
# Keep only relevant columns for join
thresholds = metric_grouped_full[['cmdb_id','kpi_name','p95','p5']].copy()

# Join per-row with its group's global p95 and p5
metric_joined = metric_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# Ensure numeric value column
metric_joined['value'] = pd.to_numeric(metric_joined['value'], errors='coerce')

# Anomaly flags based on global thresholds (p95/p5 computed globally beforehand)
metric_joined['high_anom'] = metric_joined['value'] >= metric_joined['p95']
metric_joined['low_anom']  = metric_joined['value'] <= metric_joined['p5']

# Compute per-point ratios for anomalies, guarding against zero/NaN thresholds
metric_joined['high_ratio'] = np.where(
    metric_joined['high_anom'] & metric_joined['p95'].notnull() & (metric_joined['p95'] != 0),
    metric_joined['value'] / metric_joined['p95'],
    np.nan
)
metric_joined['low_ratio'] = np.where(
    metric_joined['low_anom'] & metric_joined['p5'].notnull() & (metric_joined['p5'] != 0),
    metric_joined['value'] / metric_joined['p5'],
    np.nan
)

# Save full per-point anomaly rows to kernel variable (rows in-window that are anomalous)
metric_anomaly_points = metric_joined[metric_joined['high_anom'] | metric_joined['low_anom']].copy()

# Aggregations:
# base aggregations per group (in-window)
base_agg = metric_joined.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    total_points=('value','count'),
    max_value=('value','max'),
    min_value=('value','min')
)

# high anomaly aggregations
high_agg = metric_joined[metric_joined['high_anom']].groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    anomaly_count_high=('high_anom','sum'),
    earliest_high_ts=('ts','min'),
    max_high_ratio=('high_ratio','max')
)

# low anomaly aggregations
low_agg = metric_joined[metric_joined['low_anom']].groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    anomaly_count_low=('low_anom','sum'),
    earliest_low_ts=('ts','min'),
    min_low_ratio=('low_ratio','min')
)

# Merge aggregates
agg = base_agg.merge(high_agg, on=['cmdb_id','kpi_name'], how='left').merge(low_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts with 0, keep timestamps and ratios as NaT/NaN when absent
agg['anomaly_count_high'] = agg['anomaly_count_high'].fillna(0).astype(int)
agg['anomaly_count_low']  = agg['anomaly_count_low'].fillna(0).astype(int)

# Filter groups that have any anomalies
agg['total_anomalies'] = agg['anomaly_count_high'] + agg['anomaly_count_low']
agg_with_anoms = agg[agg['total_anomalies'] > 0].copy()

# Order by total anomaly count desc, limit to top 20
agg_with_anoms = agg_with_anoms.sort_values(['total_anomalies','total_points'], ascending=[False, False]).reset_index(drop=True)
agg_top20 = agg_with_anoms[[
    'cmdb_id','kpi_name','total_points','anomaly_count_high','anomaly_count_low',
    'earliest_high_ts','earliest_low_ts','max_value','min_value','max_high_ratio','min_low_ratio'
]].head(20)

# Final compact output: aggregated table (top groups with anomalies)
agg_top20

```
Out[3]:
```
import pandas as pd

# Summarize anomaly results using existing kernel variables:
# metric_anomaly_points (per-point anomaly rows) and agg_top20 (aggregated top-20 groups with anomalies)

total_anomaly_points = int(metric_anomaly_points.shape[0]) if 'metric_anomaly_points' in globals() else 0
unique_anomalous_groups = int(metric_anomaly_points[['cmdb_id','kpi_name']].drop_duplicates().shape[0]) if total_anomaly_points>0 else 0

# Build a short list of top groups (from agg_top20 if available)
top_list = []
if 'agg_top20' in globals() and not agg_top20.empty:
    for r in agg_top20[['cmdb_id','kpi_name','total_points','anomaly_count_high','anomaly_count_low']].head(10).itertuples(index=False):
        top_list.append(f"{r.cmdb_id}/{r.kpi_name} (points={r.total_points}, high={r.anomaly_count_high}, low={r.anomaly_count_low})")
top_groups_str = "; ".join(top_list) if top_list else "None"

summary = (
    f"Anomaly summary for metric series in window 2021-03-25 06:30:00+08:00 to 2021-03-25 07:00:00+08:00:\n\n"
    f"- Total anomaly points detected: {total_anomaly_points}\n"
    f"- Number of distinct metric series with anomalies: {unique_anomalous_groups}\n\n"
    f"- Top anomalous series (representative, top 10): {top_groups_str}\n\n"
    f"- Observations from the aggregated table (top 20 shown earlier):\n"
    f"  * Many JVM memory metrics (IG01, IG02, MG01, MG02) and several OSLinux/process metrics show anomalies with 31 points\n"
    f"    (anomaly_count_high=31 and anomaly_count_low=31), earliest anomaly timestamp = 2021-03-25 06:30:00+08:00.\n"
    f"  * Some metrics have p95 or p5 equal to observed values (ratios = 1.0). A few series have zero-valued thresholds\n"
    f"    (e.g., TCP close-wait, some MySQL response time entries), which results in NaN for ratio calculations.\n\n"
    f"- The full per-point anomaly rows are saved in the kernel variable: metric_anomaly_points\n"
    f"- The full grouped thresholds and per-group details are in metric_grouped_full.\n\n"
    f"If you want, I can: (1) show the full list of anomalous groups, (2) display per-point samples from metric_anomaly_points,\n"
    f"or (3) compute a short ranked severity score per group."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points  anomaly_count_high  anomaly_count_low          earliest_high_ts           earliest_low_ts     max_value     min_value  max_high_ratio  min_low_ratio
0      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  2.040136e+09  2.040136e+09             1.0            1.0
1      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  5.846119e+07  5.846119e+07             1.0            1.0
2      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  2.040136e+09  2.040136e+09             1.0            1.0
3      IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  5.860534e+07  5.860534e+07             1.0            1.0
4      MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  2.040136e+09  2.040136e+09             1.0            1.0
5      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  5.890014e+07  5.890014e+07             1.0            1.0
6      MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  0.000000e+00  0.000000e+00             NaN            NaN
7      MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  2.000000e+00  2.000000e+00             1.0            1.0
8      MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  3.000000e-01  3.000000e-01             1.0            1.0
9      MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  3.000000e+00  3.000000e+00             1.0            1.0
10     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  2.040136e+09  2.040136e+09             1.0            1.0
11     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  5.898395e+07  5.898395e+07             1.0            1.0
12     MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  0.000000e+00  0.000000e+00             NaN            NaN
13     MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  0.000000e+00  0.000000e+00             NaN            NaN
14     MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  1.000000e+00  1.000000e+00             1.0            1.0
15     MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  3.000000e-01  3.000000e-01             1.0            1.0
16     MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  3.000000e+00  3.000000e+00             1.0            1.0
17  Mysql01         Mysql-MySQL_3306_GetConnectedStateOfMysqld            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  1.000000e+00  1.000000e+00             1.0            1.0
18  Mysql01           Mysql-MySQL_3306_GetResponseTimeOfMysqld            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  0.000000e+00  0.000000e+00             NaN            NaN
19  Mysql02         Mysql-MySQL_3306_GetConnectedStateOfMysqld            31                  31                 31 2021-03-25 06:30:00+08:00 2021-03-25 06:30:00+08:00  1.000000e+00  1.000000e+00             1.0            1.0```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_anomaly_points exists
if 'metric_anomaly_points' not in globals() or metric_anomaly_points.empty:
    metric_faults = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_ts','fault_end_ts','duration_points',
        'anomaly_type','anomaly_count_high','anomaly_count_low','max_value','min_value',
        'p95','p5','severity_score','earliest_point_ts'
    ])
else:
    mp = metric_anomaly_points.copy()

    # Ensure ts is timezone-aware UTC+8
    mp['ts'] = pd.to_datetime(mp['ts']).dt.tz_convert(tz)

    # Ensure boolean flags exist
    for col in ['high_anom','low_anom']:
        if col not in mp.columns:
            mp[col] = False

    # Ensure numeric value
    mp['value'] = pd.to_numeric(mp['value'], errors='coerce')

    # Ensure thresholds (p95/p5) present by merging from metric_grouped_full if missing
    if not {'p95','p5'}.issubset(mp.columns):
        if 'metric_grouped_full' in globals():
            thresholds = metric_grouped_full[['cmdb_id','kpi_name','p95','p5']].copy()
            mp = mp.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
        else:
            mp['p95'] = np.nan
            mp['p5'] = np.nan

    # Compute ratios if absent
    if 'high_ratio' not in mp.columns:
        mp['high_ratio'] = np.where(
            mp['high_anom'] & mp['p95'].notnull() & (mp['p95'] != 0),
            mp['value'] / mp['p95'],
            np.nan
        )
    if 'low_ratio' not in mp.columns:
        mp['low_ratio'] = np.where(
            mp['low_anom'] & mp['p5'].notnull() & (mp['p5'] != 0),
            mp['value'] / mp['p5'],
            np.nan
        )

    # Sort for grouping
    mp = mp.sort_values(['cmdb_id','kpi_name','ts']).reset_index(drop=True)

    # Identify consecutive faults: within each (cmdb_id,kpi_name), consecutive timestamps with 60s step are same fault
    def assign_fault_ids(df):
        df = df.sort_values('ts').copy()
        df['dt'] = df['ts'].diff().dt.total_seconds().fillna(999)  # first row considered new fault
        df['new_fault'] = (df['dt'] != 60).astype(int)
        df['fault_idx'] = df['new_fault'].cumsum()
        return df

    mp = mp.groupby(['cmdb_id','kpi_name'], group_keys=False).apply(assign_fault_ids).reset_index(drop=True)

    # Aggregate per fault group
    grouped = mp.groupby(['cmdb_id','kpi_name','fault_idx'], as_index=False).agg(
        fault_start_ts=('ts', 'min'),
        fault_end_ts=('ts', 'max'),
        duration_points=('ts', 'count'),
        anomaly_count_high=('high_anom', 'sum'),
        anomaly_count_low=('low_anom', 'sum'),
        max_value=('value', 'max'),
        min_value=('value', 'min'),
        p95=('p95', 'first'),
        p5=('p5', 'first'),
        max_high_ratio=('high_ratio', 'max'),
        min_low_ratio=('low_ratio', 'min')
    )

    # Determine anomaly_type (>50% high => 'high' else 'low')
    grouped['anomaly_type'] = np.where(
        grouped['anomaly_count_high'] > (grouped['duration_points'] / 2),
        'high',
        'low'
    )

    # Compute severity_score per fault
    def compute_severity(row):
        if row['anomaly_type'] == 'high':
            if pd.notna(row['p95']) and row['p95'] != 0:
                return row['max_high_ratio'] if pd.notna(row['max_high_ratio']) else np.nan
            else:
                return np.nan
        else:
            if pd.notna(row['p5']) and row['p5'] != 0:
                return row['min_low_ratio'] if pd.notna(row['min_low_ratio']) else np.nan
            else:
                return np.nan

    grouped['severity_score'] = grouped.apply(compute_severity, axis=1)
    grouped['earliest_point_ts'] = grouped['fault_start_ts']

    # Final faults table with required columns
    metric_faults = grouped[[
        'cmdb_id','kpi_name','fault_start_ts','fault_end_ts','duration_points',
        'anomaly_type','anomaly_count_high','anomaly_count_low','max_value','min_value',
        'p95','p5','severity_score','earliest_point_ts'
    ]].copy()

# Save full faults table to kernel variable (already assigned)
# Prepare compact top-20 table ordered by duration_points desc then severity_score desc
metric_faults_top20 = metric_faults.sort_values(['duration_points','severity_score'], ascending=[False, False]).head(20)

# Return only the compact top-20 table
metric_faults_top20

```
Out[4]:
```
```python
# Summarize metric fault detection results in plain English (compact)
# Uses existing kernel variables: metric_faults, metric_faults_top20

# Safely access variables
mf = metric_faults if 'metric_faults' in globals() else None
mf_top = metric_faults_top20 if 'metric_faults_top20' in globals() else None

if mf is None or mf.empty:
    summary = "No faults detected in metric_anomaly_points within the specified window."
else:
    total_faults = int(mf.shape[0])
    unique_series = int(mf[['cmdb_id','kpi_name']].drop_duplicates().shape[0])
    # Build short list of top faults
    top_items = []
    if mf_top is not None and not mf_top.empty:
        for r in mf_top.itertuples(index=False):
            # r fields: cmdb_id,kpi_name,fault_start_ts,fault_end_ts,duration_points,...
            top_items.append(f"{r.cmdb_id}/{r.kpi_name} ({r.duration_points} pts, {r.anomaly_type}, severity={r.severity_score})")
    else:
        # fallback: take first up to 10 from full table
        sample = mf.sort_values(['duration_points','severity_score'], ascending=[False, False]).head(10)
        for r in sample.itertuples(index=False):
            top_items.append(f"{r.cmdb_id}/{r.kpi_name} ({r.duration_points} pts)")
    top_summary = "; ".join(top_items)

    # Key observations
    obs = [
        f"Total faults detected: {total_faults}",
        f"Distinct metric series with faults: {unique_series}",
        "Many faults span the entire incident window (2021-03-25 06:30:00+08:00 to 2021-03-25 07:00:00+08:00) with 31 consecutive points.",
        "Top affected components/metrics include JVM memory metrics on IG01, IG02, MG01, MG02; MySQL connection state on Mysql01/Mysql02; and several Tomcat request/thread/request-error metrics on Tomcat01.",
        "Most faults are marked 'high' (high_anom majority) and have severity_score = 1.0 where p95 equals observed values; some series with zero thresholds produce NaN severity.",
        "Full per-fault table is saved in kernel variable: metric_faults. Top 20 faults (by duration then severity) are in metric_faults_top20."
    ]

    summary = "Metric fault detection summary:\n\n" + "\n".join(obs) + "\n\nTop faults (representative, up to 10):\n" + top_summary

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name            fault_start_ts              fault_end_ts  duration_points anomaly_type  anomaly_count_high  anomaly_count_low     max_value     min_value           p95            p5  severity_score         earliest_point_ts
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09             1.0 2021-03-25 06:30:00+08:00
5         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  5.846119e+07  5.846119e+07  5.846119e+07  5.846119e+07             1.0 2021-03-25 06:30:00+08:00
519       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09             1.0 2021-03-25 06:30:00+08:00
523       IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  5.860534e+07  5.860534e+07  5.860534e+07  5.860534e+07             1.0 2021-03-25 06:30:00+08:00
1021      MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09             1.0 2021-03-25 06:30:00+08:00
1028      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  5.890014e+07  5.890014e+07  5.890014e+07  5.890014e+07             1.0 2021-03-25 06:30:00+08:00
1652      MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  2.000000e+00  2.000000e+00  2.000000e+00  2.000000e+00             1.0 2021-03-25 06:30:00+08:00
1678      MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P... 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  3.000000e-01  3.000000e-01  3.000000e-01  3.000000e-01             1.0 2021-03-25 06:30:00+08:00
1679      MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00             1.0 2021-03-25 06:30:00+08:00
1737      MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09             1.0 2021-03-25 06:30:00+08:00
1744      MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  5.898395e+07  5.898395e+07  5.898395e+07  5.898395e+07             1.0 2021-03-25 06:30:00+08:00
2363      MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00             1.0 2021-03-25 06:30:00+08:00
2388      MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P... 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  3.000000e-01  3.000000e-01  3.000000e-01  3.000000e-01             1.0 2021-03-25 06:30:00+08:00
2389      MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00             1.0 2021-03-25 06:30:00+08:00
2481   Mysql01         Mysql-MySQL_3306_GetConnectedStateOfMysqld 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00             1.0 2021-03-25 06:30:00+08:00
3287   Mysql02         Mysql-MySQL_3306_GetConnectedStateOfMysqld 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00             1.0 2021-03-25 06:30:00+08:00
6525  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun... 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  1.051000e+03  1.051000e+03  1.051000e+03  1.051000e+03             1.0 2021-03-25 06:30:00+08:00
6526  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe... 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  3.978080e+05  3.978080e+05  3.978080e+05  3.978080e+05             1.0 2021-03-25 06:30:00+08:00
6540  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr... 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02             1.0 2021-03-25 06:30:00+08:00
6542  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads... 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               31         high                  31                 31  5.000000e+02  5.000000e+02  5.000000e+02  5.000000e+02             1.0 2021-03-25 06:30:00+08:00```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse existing metric_faults variable (full per-fault table)
# Ensure metric_faults exists
if 'metric_faults' not in globals():
    metric_faults_by_component = pd.DataFrame(columns=[
        'cmdb_id','fault_count','distinct_kpi_count','earliest_fault_ts',
        'latest_fault_ts','total_duration_points','max_severity_score'
    ])
else:
    mf = metric_faults.copy()

    # Ensure timestamp columns are timezone-aware UTC+8
    for col in ['fault_start_ts','fault_end_ts','earliest_point_ts']:
        if col in mf.columns:
            mf[col] = pd.to_datetime(mf[col]).dt.tz_convert(tz)

    # Ensure numeric severity_score and duration_points
    mf['severity_score'] = pd.to_numeric(mf['severity_score'], errors='coerce')
    mf['duration_points'] = pd.to_numeric(mf['duration_points'], errors='coerce').fillna(0).astype(int)

    # Aggregate by cmdb_id
    metric_faults_by_component = mf.groupby('cmdb_id', as_index=False).agg(
        fault_count = ('kpi_name', 'count'),
        distinct_kpi_count = ('kpi_name', 'nunique'),
        earliest_fault_ts = ('fault_start_ts', 'min'),
        latest_fault_ts = ('fault_end_ts', 'max'),
        total_duration_points = ('duration_points', 'sum'),
        max_severity_score = ('severity_score', 'max')
    )

    # Order by fault_count desc then max_severity_score desc
    metric_faults_by_component = metric_faults_by_component.sort_values(
        ['fault_count','max_severity_score'], ascending=[False, False]
    ).reset_index(drop=True)

# Save full aggregated table to kernel variable and return top 20 rows (compact)
metric_faults_by_component_top20 = metric_faults_by_component.head(20)
metric_faults_by_component, metric_faults_by_component_top20

```
Out[5]:
```
Summary (faults aggregated by component, window 2021-03-25 06:30–07:00 UTC+8):

- Top components by number of faults (showing: fault_count, distinct_kpi_count, total_duration_points, max_severity_score):
  - Redis02: 872 faults, 109 distinct KPIs, total_duration_points=1279, max_severity_score=2.2727
  - Mysql02: 838 faults, 223 KPIs, total_duration_points=3037, max_severity_score=1.6667
  - Mysql01: 807 faults, 224 KPIs, total_duration_points=3371, max_severity_score=2.0000
  - Redis01: 801 faults, 112 KPIs, total_duration_points=1078, max_severity_score=1.7324
  - Tomcat01: 782 faults, 117 KPIs, total_duration_points=1397, max_severity_score=3.3333
  - Tomcat04: 760 faults, 122 KPIs, total_duration_points=1466, max_severity_score=2.8571
  - Tomcat03: 740 faults, 120 KPIs, total_duration_points=1445, max_severity_score=4.3686
  - Tomcat02: 723 faults, 121 KPIs, total_duration_points=1429, max_severity_score=2.8571
  - MG01: 716 faults, 105 KPIs, total_duration_points=985, max_severity_score=2.8571
  - MG02: 706 faults, 104 KPIs, total_duration_points=960,  max_severity_score=2.8571
  - (Others in top 13: apache01, IG01, apache02, IG02 with lower counts but non-trivial severity scores)

- Time coverage: earliest_fault_ts and latest_fault_ts for all listed components fall inside the incident window (2021-03-25 06:30:00+08:00 through 2021-03-25 07:00:00+08:00).

- Interpretation:
  - Redis and MySQL components show the highest number of distinct faults and the largest total_duration_points, indicating broad and sustained anomalous behavior across many KPIs during the window.
  - Several Tomcat components show high max_severity_score values (Tomcat03 ≈4.37, Tomcat01 ≈3.33), indicating some faults with values multiple times above the global p95 (higher severity).
  - Many components have faults across many distinct KPIs, suggesting system-wide disturbance rather than isolated single-metric spikes.

- Data availability: The full aggregated per-component table is saved in the kernel variable metric_faults_by_component. If you want, I can:
  - show the complete table,
  - filter components by severity or KPI count,
  - or produce a short prioritized list of components to investigate first (e.g., Redis02, Mysql02, Mysql01, Tomcat03, Tomcat01).

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  fault_count  distinct_kpi_count         earliest_fault_ts           latest_fault_ts  total_duration_points  max_severity_score
0    Redis02          872                 109 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   1279            2.272727
1    Mysql02          838                 223 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   3037            1.666667
2    Mysql01          807                 224 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   3371            2.000000
3    Redis01          801                 112 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   1078            1.732393
4   Tomcat01          782                 117 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   1397            3.333333
5   Tomcat04          760                 122 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   1466            2.857143
6   Tomcat03          740                 120 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   1445            4.368557
7   Tomcat02          723                 121 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   1429            2.857143
8       MG01          716                 105 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                    985            2.857143
9       MG02          706                 104 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                    960            2.857143
10  apache01          530                  80 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                    625            1.987133
11      IG01          519                  88 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                    661            1.666667
12  apache02          506                  77 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                    598            1.818182
13      IG02          502                  87 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                    639            1.657874,      cmdb_id  fault_count  distinct_kpi_count         earliest_fault_ts           latest_fault_ts  total_duration_points  max_severity_score
0    Redis02          872                 109 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   1279            2.272727
1    Mysql02          838                 223 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   3037            1.666667
2    Mysql01          807                 224 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   3371            2.000000
3    Redis01          801                 112 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   1078            1.732393
4   Tomcat01          782                 117 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   1397            3.333333
5   Tomcat04          760                 122 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   1466            2.857143
6   Tomcat03          740                 120 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   1445            4.368557
7   Tomcat02          723                 121 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                   1429            2.857143
8       MG01          716                 105 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                    985            2.857143
9       MG02          706                 104 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                    960            2.857143
10  apache01          530                  80 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                    625            1.987133
11      IG01          519                  88 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                    661            1.666667
12  apache02          506                  77 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                    598            1.818182
13      IG02          502                  87 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                    639            1.657874)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import re
import pytz

# Reuse existing kernel variables: trace_df, trace_grouped_full
tz = pytz.timezone('Asia/Shanghai')

# Ensure trace_df has timezone-aware 'ts'
trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window (inclusive)
window_start = pd.Timestamp('2021-03-25 06:30:00', tz=tz)
window_end   = pd.Timestamp('2021-03-25 07:00:00', tz=tz)

# Filter trace rows to window
trace_window = trace_df[(trace_df['ts'] >= window_start) & (trace_df['ts'] <= window_end)].copy()

# Prepare thresholds lookup from trace_grouped_full
thresholds = trace_grouped_full[['cmdb_id','trace_name','p95','p5']].copy()

# Join thresholds
trace_joined = trace_window.merge(thresholds, on=['cmdb_id','trace_name'], how='left')

# Ensure numeric value
trace_joined['value'] = pd.to_numeric(trace_joined['value'], errors='coerce')

# Flag anomalies
trace_joined['high_anom'] = trace_joined['value'] >= trace_joined['p95']
trace_joined['low_anom']  = trace_joined['value'] <= trace_joined['p5']

# Keep only anomalous rows (either high or low)
trace_anomaly_points = trace_joined[trace_joined['high_anom'] | trace_joined['low_anom']].copy()

# Parse trace_name to extract caller/callee
def parse_peer(row):
    name = row['trace_name'] if pd.notna(row['trace_name']) else ''
    cmdb = row['cmdb_id']
    # trace.to_X -> caller = cmdb, callee = X
    m_to = re.search(r'trace\.to_([^\.]+)', name)
    if m_to:
        return pd.Series({'caller': cmdb, 'callee': m_to.group(1)})
    # trace.from_X -> caller = X, callee = cmdb
    m_from = re.search(r'trace\.from_([^\.]+)', name)
    if m_from:
        return pd.Series({'caller': m_from.group(1), 'callee': cmdb})
    # trace.self.* -> peer None
    if name.startswith('trace.self'):
        return pd.Series({'caller': None, 'callee': None})
    # Fallback: unknown -> caller=None, callee=None
    return pd.Series({'caller': None, 'callee': None})

trace_anomaly_points[['caller','callee']] = trace_anomaly_points.apply(parse_peer, axis=1)

# Determine anomaly_type per row: 'high' if high_anom >= low_anom else 'low'
# (booleans cast to ints; handles both True case by preferring 'high')
trace_anomaly_points['anomaly_type'] = np.where(
    trace_anomaly_points['high_anom'].astype(int) >= trace_anomaly_points['low_anom'].astype(int),
    'high','low'
)

# Compute per-row severity: for high -> value/p95 (NaN if p95<=0 or NaN); for low -> value/p5 (NaN if p5<=0 or NaN)
trace_anomaly_points['severity'] = np.nan
# high
mask_high = trace_anomaly_points['anomaly_type']=='high'
mask_high_valid = mask_high & trace_anomaly_points['p95'].notna() & (trace_anomaly_points['p95'] != 0)
trace_anomaly_points.loc[mask_high_valid, 'severity'] = trace_anomaly_points.loc[mask_high_valid, 'value'] / trace_anomaly_points.loc[mask_high_valid, 'p95']
# low
mask_low = trace_anomaly_points['anomaly_type']=='low'
mask_low_valid = mask_low & trace_anomaly_points['p5'].notna() & (trace_anomaly_points['p5'] != 0)
trace_anomaly_points.loc[mask_low_valid, 'severity'] = trace_anomaly_points.loc[mask_low_valid, 'value'] / trace_anomaly_points.loc[mask_low_valid, 'p5']

# Save per-point anomalous trace rows to kernel variable
# (full contents kept in memory; don't print)
# trace_anomaly_points variable is already assigned

# Aggregate anomalous trace rows by (caller, callee)
agg = trace_anomaly_points.groupby(['caller','callee'], dropna=False).agg(
    anomaly_count = ('severity','count'),
    earliest_anom_ts = ('ts','min'),
    latest_anom_ts = ('ts','max'),
    max_severity = ('severity','max')
).reset_index()

# Sort by anomaly_count desc then max_severity desc and keep top 20
trace_anomaly_pairs = agg.sort_values(['anomaly_count','max_severity'], ascending=[False, False]).reset_index(drop=True)
trace_anomaly_pairs_top20 = trace_anomaly_pairs[['caller','callee','anomaly_count','earliest_anom_ts','latest_anom_ts','max_severity']].head(20)

# Save full aggregated pairs to kernel variable trace_anomaly_pairs (already assigned)
trace_anomaly_pairs, trace_anomaly_pairs_top20

```
Out[6]:
```
Summary of anomalous trace analysis for window 2021-03-25 06:30–07:00 (UTC+8)

What I did (brief):
- Joined per-point trace rows with globally computed p95/p5 from trace_grouped_full.
- Flagged rows where value >= p95 (high_anom) or value <= p5 (low_anom).
- Parsed trace_name to infer caller/callee:
  - trace.to_X => caller = current cmdb_id, callee = X
  - trace.from_X => caller = X, callee = current cmdb_id
  - trace.self.* => caller/callee = None
- Saved per-point anomalous trace rows to kernel variable: trace_anomaly_points.
- Aggregated anomalous rows by (caller, callee) and saved that table to trace_anomaly_pairs. Returned top pairs ordered by anomaly_count then max_severity.

Key results (top pairs — anomaly_count, earliest, latest, max_severity)
- caller=None / callee=None (trace.self.*): anomaly_count=98, earliest=06:30, latest=07:00, max_severity≈2.01
  - Many self-trace anomalies exist (no peer), highest overall anomaly count.
- Tomcat01 → MG02: anomaly_count=32, earliest=06:30, latest=06:52, max_severity≈1.463
- IG01 → Tomcat01: anomaly_count=28, earliest=06:33, latest=06:56, max_severity≈2.756
- dockerA1 → MG01: anomaly_count=26, earliest=06:31, latest=06:59, max_severity≈3.666
- Tomcat03 → MG02: anomaly_count=26, earliest=06:30, latest=06:58, max_severity≈1.674
- Additional notable pairs (examples): IG02→Tomcat03, IG01→Tomcat03, Tomcat03→MG01, Tomcat02→MG02, Tomcat04→MG01, IG02→Tomcat02, IG01→Tomcat02, IG01→Tomcat04, Tomcat01→MG01, IG02→Tomcat01, Tomcat04→MG02, Tomcat02→MG01, MG02→dockerB2, dockerA1→MG02 (the latter shows very high max_severity ≈10.24 for fewer points).

Observations / interpretation:
- Most anomalous trace activity is concentrated in the incident window 06:30–07:00 (all earliest/latest timestamps fall within this range).
- The largest group of anomalies are trace.self.* entries (caller/callee None) — these represent local service trace metrics that deviated from global thresholds.
- Several service-to-service pairs involving Tomcat nodes and MG/IG components show sustained anomalies (dozens of anomalous points), suggesting widespread inter-service performance deviations.
- Some caller→callee pairs have high severity (severity > 3), e.g., dockerA1→MG01 and dockerA1→MG02 (max severity ≈3.67 and ≈10.24 respectively), indicating individual points that greatly exceed global p95 thresholds.
- The combination of high anomaly counts and elevated severities on Tomcat↔MG/IG paths points to performance issues affecting service-to-service interactions as well as local service (self) metrics.

Data saved in kernel:
- Per-point anomalous trace rows: trace_anomaly_points
- Aggregated anomalous pairs: trace_anomaly_pairs

Next useful steps (optional):
- Inspect trace_anomaly_points samples for the highest-severity pairs (e.g., dockerA1→MG02) to see exact timestamps and values.
- Correlate the top anomalous trace pairs with metric and log anomalies on the involved components (Tomcat/MG/IG/docker nodes) to find likely root causes.
- Drill into trace.self.* anomalies per component to see whether local latency or error counts spiked concurrently.

Tell me which follow-up you want (show samples for a specific pair, filter by severity threshold, or correlate with metric/log anomalies).

The original code execution output of IPython Kernel is also provided below for reference:

(      caller    callee  anomaly_count          earliest_anom_ts            latest_anom_ts  max_severity
0        NaN       NaN             98 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00      2.010692
1   Tomcat01      MG02             32 2021-03-25 06:30:00+08:00 2021-03-25 06:52:00+08:00      1.462719
2       IG01  Tomcat01             28 2021-03-25 06:33:00+08:00 2021-03-25 06:56:00+08:00      2.756231
3   dockerA1      MG01             26 2021-03-25 06:31:00+08:00 2021-03-25 06:59:00+08:00      3.665543
4   Tomcat03      MG02             26 2021-03-25 06:30:00+08:00 2021-03-25 06:58:00+08:00      1.674321
..       ...       ...            ...                       ...                       ...           ...
28      MG02  dockerB1             13 2021-03-25 06:31:00+08:00 2021-03-25 06:59:00+08:00      1.384182
29      MG01  dockerA1             13 2021-03-25 06:30:00+08:00 2021-03-25 06:56:00+08:00      1.288584
30  dockerB1      MG02             12 2021-03-25 06:32:00+08:00 2021-03-25 06:57:00+08:00      3.223634
31      MG01  dockerB2             12 2021-03-25 06:31:00+08:00 2021-03-25 07:00:00+08:00      2.179042
32  dockerB1      MG01             12 2021-03-25 06:33:00+08:00 2021-03-25 06:59:00+08:00      1.738947

[33 rows x 6 columns],       caller    callee  anomaly_count          earliest_anom_ts            latest_anom_ts  max_severity
0        NaN       NaN             98 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00      2.010692
1   Tomcat01      MG02             32 2021-03-25 06:30:00+08:00 2021-03-25 06:52:00+08:00      1.462719
2       IG01  Tomcat01             28 2021-03-25 06:33:00+08:00 2021-03-25 06:56:00+08:00      2.756231
3   dockerA1      MG01             26 2021-03-25 06:31:00+08:00 2021-03-25 06:59:00+08:00      3.665543
4   Tomcat03      MG02             26 2021-03-25 06:30:00+08:00 2021-03-25 06:58:00+08:00      1.674321
5       IG02  Tomcat03             26 2021-03-25 06:30:00+08:00 2021-03-25 06:52:00+08:00      1.483683
6       IG01  Tomcat03             26 2021-03-25 06:30:00+08:00 2021-03-25 06:58:00+08:00      1.247237
7   Tomcat03      MG01             26 2021-03-25 06:33:00+08:00 2021-03-25 06:57:00+08:00      1.130435
8   Tomcat02      MG02             24 2021-03-25 06:35:00+08:00 2021-03-25 06:59:00+08:00      2.961751
9   Tomcat04      MG01             24 2021-03-25 06:30:00+08:00 2021-03-25 06:59:00+08:00      2.102719
10      IG02  Tomcat02             24 2021-03-25 06:35:00+08:00 2021-03-25 07:00:00+08:00      2.004111
11      IG01  Tomcat02             24 2021-03-25 06:33:00+08:00 2021-03-25 06:55:00+08:00      1.965363
12      IG01  Tomcat04             24 2021-03-25 06:32:00+08:00 2021-03-25 06:59:00+08:00      1.941075
13  Tomcat01      MG01             24 2021-03-25 06:31:00+08:00 2021-03-25 06:57:00+08:00      1.739165
14      IG02  Tomcat01             24 2021-03-25 06:32:00+08:00 2021-03-25 06:53:00+08:00      1.522176
15  Tomcat04      MG02             24 2021-03-25 06:31:00+08:00 2021-03-25 06:57:00+08:00      1.478777
16  Tomcat02      MG01             24 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00      1.413055
17      IG02  Tomcat04             24 2021-03-25 06:32:00+08:00 2021-03-25 06:59:00+08:00      1.245301
18      MG02  dockerB2             20 2021-03-25 06:30:00+08:00 2021-03-25 06:56:00+08:00      1.365890
19  dockerA1      MG02             17 2021-03-25 06:36:00+08:00 2021-03-25 06:59:00+08:00     10.243236)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables: trace_anomaly_pairs, metric_faults_by_component
pairs = trace_anomaly_pairs.copy() if 'trace_anomaly_pairs' in globals() else pd.DataFrame()
mf_comp = metric_faults_by_component.copy() if 'metric_faults_by_component' in globals() else pd.DataFrame()

# Normalize timestamp columns to UTC+8 tz if present
for col in ['earliest_anom_ts','latest_anom_ts']:
    if col in pairs.columns:
        pairs[col] = pd.to_datetime(pairs[col]).dt.tz_convert(tz)

# 1) Keep only rows where callee is not null
pairs_nonnull = pairs[pairs['callee'].notna()].copy()

# 1a) Compute callee_has_metric_fault by checking presence in metric_faults_by_component.cmdb_id
mf_components = set(mf_comp['cmdb_id'].unique()) if not mf_comp.empty else set()
pairs_nonnull['callee_has_metric_fault'] = pairs_nonnull['callee'].isin(mf_components)

# 2) Aggregate by callee
agg = pairs_nonnull.groupby('callee', as_index=False).agg(
    callee_has_metric_fault = ('callee_has_metric_fault', 'any'),
    num_distinct_callers = ('caller', lambda x: x.nunique(dropna=True)),
    total_incoming_anomaly_count = ('anomaly_count', 'sum'),
    earliest_incoming_anom_ts = ('earliest_anom_ts', 'min'),
    latest_incoming_anom_ts = ('latest_anom_ts', 'max'),
    max_incoming_severity = ('max_severity', 'max')
)

# Ensure boolean column is True/False
agg['callee_has_metric_fault'] = agg['callee_has_metric_fault'].astype(bool)

# 3) Order as requested: callee_has_metric_fault desc, then total_incoming_anomaly_count desc, then max_incoming_severity desc
trace_localized_candidates = agg.sort_values(
    by=['callee_has_metric_fault','total_incoming_anomaly_count','max_incoming_severity'],
    ascending=[False, False, False]
).reset_index(drop=True)

# 4) Save full aggregated result to kernel variable trace_localized_candidates and return only top 20 rows (compact)
trace_localized_candidates_top20 = trace_localized_candidates[[
    'callee','callee_has_metric_fault','num_distinct_callers',
    'total_incoming_anomaly_count','earliest_incoming_anom_ts',
    'latest_incoming_anom_ts','max_incoming_severity'
]].head(20)

trace_localized_candidates, trace_localized_candidates_top20

```
Out[7]:
```
summary = (
    "Trace-localized candidate components (window 2021-03-25 06:30–07:00 UTC+8):\n\n"
    "Top callees receiving anomalous traces:\n"
    "1) MG01 — callee_has_metric_fault: True; distinct_callers=8; total_incoming_anomaly_count=169; "
    "earliest=2021-03-25 06:30; latest=2021-03-25 07:00; max_incoming_severity≈17.43\n"
    "2) MG02 — callee_has_metric_fault: True; distinct_callers=8; total_incoming_anomaly_count=165; "
    "earliest=2021-03-25 06:30; latest=2021-03-25 06:59; max_incoming_severity≈10.24\n"
    "3) Tomcat01 — callee_has_metric_fault: True; distinct_callers=2; total_incoming_anomaly_count=52; "
    "earliest=2021-03-25 06:32; latest=2021-03-25 06:56; max_incoming_severity≈2.76\n"
    "4) Tomcat03 — callee_has_metric_fault: True; distinct_callers=2; total_incoming_anomaly_count=52; "
    "earliest=2021-03-25 06:30; latest=2021-03-25 06:58; max_incoming_severity≈1.48\n"
    "5) Tomcat02 — callee_has_metric_fault: True; distinct_callers=2; total_incoming_anomaly_count=48; "
    "earliest=2021-03-25 06:33; latest=2021-03-25 07:00; max_incoming_severity≈2.00\n"
    "6) Tomcat04 — callee_has_metric_fault: True; distinct_callers=2; total_incoming_anomaly_count=48; "
    "earliest=2021-03-25 06:32; latest=2021-03-25 06:59; max_incoming_severity≈1.94\n"
    "7) dockerB2 — callee_has_metric_fault: False; distinct_callers=2; total_incoming_anomaly_count=32; "
    "earliest=2021-03-25 06:30; latest=2021-03-25 07:00; max_incoming_severity≈2.18\n"
    "8) dockerA1 — callee_has_metric_fault: False; distinct_callers=2; total_incoming_anomaly_count=29; "
    "earliest=2021-03-25 06:30; latest=2021-03-25 06:59; max_incoming_severity≈1.49\n"
    "9) dockerA2 — callee_has_metric_fault: False; distinct_callers=2; total_incoming_anomaly_count=26; "
    "earliest=2021-03-25 06:30; latest=2021-03-25 06:58; max_incoming_severity≈4.02\n"
    "10) dockerB1 — callee_has_metric_fault: False; distinct_callers=2; total_incoming_anomaly_count=26; "
    "earliest=2021-03-25 06:31; latest=2021-03-25 06:59; max_incoming_severity≈1.74\n\n"
    "Interpretation and next steps:\n"
    "- MG01 and MG02 are the top candidates: they receive anomalies from many callers (8 each), have metric faults, "
    "and very high max severities (MG01 ≈17.4, MG02 ≈10.2). Investigate MG01/MG02 first.\n"
    "- Several Tomcat nodes (Tomcat01/02/03/04) also appear as callees and have metric faults — investigate these next, "
    "especially Tomcat01 (higher severity ~2.76).\n"
    "- Docker hosts (dockerA1/A2/B1/B2) show anomalous incoming traces but do not have metric faults recorded in metrics; "
    "they may be callers causing downstream load or represent instrumentation differences.\n"
    "- The aggregated candidate table is saved in kernel variable: trace_localized_candidates. Per-pair details are in trace_anomaly_pairs.\n\n"
    "If you want, I can (a) show the full candidate table, (b) filter to only callees with metric faults, or (c) "
    "correlate these candidates with metric_faults_by_component time ranges or top KPIs."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     callee  callee_has_metric_fault  num_distinct_callers  total_incoming_anomaly_count earliest_incoming_anom_ts   latest_incoming_anom_ts  max_incoming_severity
0      MG01                     True                     8                           169 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00              17.434955
1      MG02                     True                     8                           165 2021-03-25 06:30:00+08:00 2021-03-25 06:59:00+08:00              10.243236
2  Tomcat01                     True                     2                            52 2021-03-25 06:32:00+08:00 2021-03-25 06:56:00+08:00               2.756231
3  Tomcat03                     True                     2                            52 2021-03-25 06:30:00+08:00 2021-03-25 06:58:00+08:00               1.483683
4  Tomcat02                     True                     2                            48 2021-03-25 06:33:00+08:00 2021-03-25 07:00:00+08:00               2.004111
5  Tomcat04                     True                     2                            48 2021-03-25 06:32:00+08:00 2021-03-25 06:59:00+08:00               1.941075
6  dockerB2                    False                     2                            32 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               2.179042
7  dockerA1                    False                     2                            29 2021-03-25 06:30:00+08:00 2021-03-25 06:59:00+08:00               1.494698
8  dockerA2                    False                     2                            26 2021-03-25 06:30:00+08:00 2021-03-25 06:58:00+08:00               4.023491
9  dockerB1                    False                     2                            26 2021-03-25 06:31:00+08:00 2021-03-25 06:59:00+08:00               1.739084,      callee  callee_has_metric_fault  num_distinct_callers  total_incoming_anomaly_count earliest_incoming_anom_ts   latest_incoming_anom_ts  max_incoming_severity
0      MG01                     True                     8                           169 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00              17.434955
1      MG02                     True                     8                           165 2021-03-25 06:30:00+08:00 2021-03-25 06:59:00+08:00              10.243236
2  Tomcat01                     True                     2                            52 2021-03-25 06:32:00+08:00 2021-03-25 06:56:00+08:00               2.756231
3  Tomcat03                     True                     2                            52 2021-03-25 06:30:00+08:00 2021-03-25 06:58:00+08:00               1.483683
4  Tomcat02                     True                     2                            48 2021-03-25 06:33:00+08:00 2021-03-25 07:00:00+08:00               2.004111
5  Tomcat04                     True                     2                            48 2021-03-25 06:32:00+08:00 2021-03-25 06:59:00+08:00               1.941075
6  dockerB2                    False                     2                            32 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00               2.179042
7  dockerA1                    False                     2                            29 2021-03-25 06:30:00+08:00 2021-03-25 06:59:00+08:00               1.494698
8  dockerA2                    False                     2                            26 2021-03-25 06:30:00+08:00 2021-03-25 06:58:00+08:00               4.023491
9  dockerB1                    False                     2                            26 2021-03-25 06:31:00+08:00 2021-03-25 06:59:00+08:00               1.739084)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables: metric_faults, metric_grouped_full
# Filter metric_faults to MG01 and MG02
mf = metric_faults if 'metric_faults' in globals() else pd.DataFrame()
mf = mf[mf['cmdb_id'].isin(['MG01','MG02'])].copy()

# Ensure timestamp cols are timezone-aware UTC+8
for col in ['fault_start_ts','fault_end_ts','earliest_point_ts']:
    if col in mf.columns:
        mf[col] = pd.to_datetime(mf[col]).dt.tz_convert(tz)

# Ensure numeric columns
mf['duration_points'] = pd.to_numeric(mf['duration_points'], errors='coerce').fillna(0).astype(int)
mf['severity_score'] = pd.to_numeric(mf['severity_score'], errors='coerce')
mf['max_value'] = pd.to_numeric(mf['max_value'], errors='coerce')
mf['min_value'] = pd.to_numeric(mf['min_value'], errors='coerce')

# Prepare thresholds lookup from metric_grouped_full (global thresholds precomputed)
mgf = metric_grouped_full[['cmdb_id','kpi_name','p95','p5']].copy() if 'metric_grouped_full' in globals() else pd.DataFrame()

# Aggregate by cmdb_id and kpi_name
def mode_series(s):
    if s.empty:
        return np.nan
    modes = s.mode(dropna=True)
    return modes.iloc[0] if not modes.empty else s.iloc[0]

agg = mf.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    fault_count = ('kpi_name','count'),
    distinct_faults_duration_points = ('duration_points','sum'),
    earliest_fault_start = ('fault_start_ts','min'),
    latest_fault_end = ('fault_end_ts','max'),
    anomaly_type_majority = ('anomaly_type', lambda x: mode_series(x.astype(str))),
    max_severity = ('severity_score','max'),
    max_value = ('max_value','max'),
    min_value = ('min_value','min')
)

# Join global p95/p5 thresholds from metric_grouped_full
if not mgf.empty:
    agg = agg.merge(mgf, on=['cmdb_id','kpi_name'], how='left')

# Save full aggregated summary to kernel variable
mg_metric_faults_summary = agg.copy()

# Prepare compact top-20 ordered by fault_count desc then max_severity desc
mg_metric_faults_summary_top20 = mg_metric_faults_summary.sort_values(
    ['fault_count','max_severity'], ascending=[False, False]
).head(20)[[
    'cmdb_id','kpi_name','fault_count','distinct_faults_duration_points',
    'earliest_fault_start','latest_fault_end','anomaly_type_majority',
    'max_severity','p95','p5','max_value','min_value'
]]

# Display compact top-20 table
mg_metric_faults_summary_top20

```
Out[8]:
```
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use existing kernel variable mg_metric_faults_summary (full aggregated summary for MG01/MG02)
df = mg_metric_faults_summary if 'mg_metric_faults_summary' in globals() else pd.DataFrame()

if df.empty:
    summary = "No metric fault summary available for MG01/MG02."
else:
    # Ensure timestamps formatted compactly
    for col in ['earliest_fault_start','latest_fault_end']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col]).dt.tz_convert(tz)
    # Counts per component
    mg01_count = int(df[df['cmdb_id']=='MG01'].shape[0])
    mg02_count = int(df[df['cmdb_id']=='MG02'].shape[0])
    # Top KPIs by fault_count for each MG
    top_mg01 = df[df['cmdb_id']=='MG01'].sort_values(['fault_count','max_severity'], ascending=[False, False]).head(5)
    top_mg02 = df[df['cmdb_id']=='MG02'].sort_values(['fault_count','max_severity'], ascending=[False, False]).head(5)
    def fmt_top(dfpart):
        if dfpart.empty:
            return "None"
        items = []
        for r in dfpart.itertuples(index=False):
            items.append(f"{r.kpi_name} (faults={r.fault_count}, dur_pts={r.distinct_faults_duration_points}, type={r.anomaly_type_majority}, max_sev={round(r.max_severity,3) if pd.notna(r.max_severity) else 'NaN'})")
        return "; ".join(items)
    top_mg01_s = fmt_top(top_mg01)
    top_mg02_s = fmt_top(top_mg02)
    # Earliest and latest overall in the summaries
    earliest = df['earliest_fault_start'].min()
    latest = df['latest_fault_end'].max()
    summary = (
        f"MG01/MG02 metric-fault summary (window 2021-03-25 06:30–07:00 UTC+8):\n\n"
        f"- Number of distinct KPI groups summarized: MG01={mg01_count}, MG02={mg02_count}.\n"
        f"- Overall time coverage in this summary: earliest fault start = {earliest}, latest fault end = {latest}.\n\n"
        f"- Top MG01 KPIs by fault count (top 5): {top_mg01_s}\n\n"
        f"- Top MG02 KPIs by fault count (top 5): {top_mg02_s}\n\n"
        "Key observations:\n"
        "- Both MG01 and MG02 show many KPI faults across filesystem, swap, disk, and process metrics.\n"
        "- Many filesystem- and swap-related KPIs for MG01/MG02 are marked 'high' and have consistent fault durations.\n"
        "- Several disk I/O and filesystem KPIs show 'low' majority with elevated max_severity values (indicating values significantly outside global thresholds).\n"
        "- p95/p5 thresholds exist in the summary where available; some KPIs have zero thresholds leading to NaN severity in calculations.\n\n"
        "The full per-KPI aggregated summary for MG01/MG02 is saved in kernel variable: mg_metric_faults_summary."
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  fault_count  distinct_faults_duration_points      earliest_fault_start          latest_fault_end anomaly_type_majority  max_severity        p95         p5  max_value  min_value
95     MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMemPerc           16                               16 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                  high           NaN   0.000000   0.000000   0.000000   0.000000
99     MG01    OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent           16                               16 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                  high           NaN   0.000000   0.000000   0.000000   0.000000
100    MG01                       OSLinux-OSLinux_SWAP_SWAP_Si           16                               16 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                  high           NaN   0.000000   0.000000   0.000000   0.000000
101    MG01                       OSLinux-OSLinux_SWAP_SWAP_So           16                               16 2021-03-25 06:30:00+08:00 2021-03-25 07:00:00+08:00                  high           NaN   0.000000   0.000000   0.000000   0.000000
200    MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMemPerc           15                               15 2021-03-25 06:31:00+08:00 2021-03-25 06:59:00+08:00                  high           NaN   0.000000   0.000000   0.000000   0.000000
203    MG02    OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent           15                               15 2021-03-25 06:31:00+08:00 2021-03-25 06:59:00+08:00                  high           NaN   0.000000   0.000000   0.000000   0.000000
204    MG02                       OSLinux-OSLinux_SWAP_SWAP_Si           15                               15 2021-03-25 06:31:00+08:00 2021-03-25 06:59:00+08:00                  high           NaN   0.000000   0.000000   0.000000   0.000000
205    MG02                       OSLinux-OSLinux_SWAP_SWAP_So           15                               15 2021-03-25 06:31:00+08:00 2021-03-25 06:59:00+08:00                  high           NaN   0.000000   0.000000   0.000000   0.000000
61     MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps           14                               14 2021-03-25 06:31:00+08:00 2021-03-25 06:57:00+08:00                   low      2.857143   0.029155   0.000000   0.083300   0.000000
62     MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead           14                               14 2021-03-25 06:31:00+08:00 2021-03-25 06:57:00+08:00                   low      2.857143   0.513345   0.000000   1.466700   0.000000
174    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps           14                               14 2021-03-25 06:31:00+08:00 2021-03-25 06:57:00+08:00                   low      2.857143   0.350000   0.000000   1.000000   0.000000
21     MG01         OSLinux-OSLinux_FILESYSTEM_----_FSCapacity           14                               14 2021-03-25 06:31:00+08:00 2021-03-25 06:57:00+08:00                   low      1.000000  30.272445  30.272418  30.272445  30.272418
22     MG01  OSLinux-OSLinux_FILESYSTEM_----_FSInodeUsedPer...           14                               14 2021-03-25 06:31:00+08:00 2021-03-25 06:57:00+08:00                  high      1.000000  16.366475  16.366475  16.366475  16.366475
23     MG01     OSLinux-OSLinux_FILESYSTEM_----boot_FSCapacity           14                               14 2021-03-25 06:31:00+08:00 2021-03-25 06:57:00+08:00                  high      1.000000  15.416955  15.416955  15.416955  15.416955
24     MG01  OSLinux-OSLinux_FILESYSTEM_----boot_FSInodeUse...           14                               14 2021-03-25 06:31:00+08:00 2021-03-25 06:57:00+08:00                  high      1.000000   0.471497   0.471497   0.471497   0.471497
25     MG01  OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSCa...           14                               14 2021-03-25 06:31:00+08:00 2021-03-25 06:57:00+08:00                  high      1.000000   0.426812   0.426812   0.426812   0.426812
26     MG01  OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSIn...           14                               14 2021-03-25 06:31:00+08:00 2021-03-25 06:57:00+08:00                  high      1.000000   0.004196   0.004196   0.004196   0.004196
28     MG01  OSLinux-OSLinux_FILESYSTEM_----home_FSInodeUse...           14                               14 2021-03-25 06:31:00+08:00 2021-03-25 06:57:00+08:00                  high      1.000000   0.222626   0.222626   0.222626   0.222626
29     MG01      OSLinux-OSLinux_FILESYSTEM_----tmp_FSCapacity           14                               14 2021-03-25 06:31:00+08:00 2021-03-25 06:57:00+08:00                  high      1.000000   0.958370   0.958370   0.958370   0.958370
30     MG01  OSLinux-OSLinux_FILESYSTEM_----tmp_FSInodeUsed...           14                               14 2021-03-25 06:31:00+08:00 2021-03-25 06:57:00+08:00                  high      1.000000   0.015869   0.015869   0.015869   0.015869```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables: metric_faults, metric_grouped_full, metric_anomaly_points
mf = metric_faults.copy() if 'metric_faults' in globals() else pd.DataFrame()
mg_full = metric_grouped_full.copy() if 'metric_grouped_full' in globals() else pd.DataFrame()
mp = metric_anomaly_points.copy() if 'metric_anomaly_points' in globals() else pd.DataFrame()

# Ensure we have timestamps tz-aware
for col in ['fault_start_ts','fault_end_ts','earliest_point_ts','ts']:
    if col in mf.columns:
        mf[col] = pd.to_datetime(mf[col]).dt.tz_convert(tz)
    if col in mp.columns:
        mp['ts'] = pd.to_datetime(mp['ts']).dt.tz_convert(tz)  # only 'ts' expected in mp

# Filter faults to MG01 and MG02
mf_mg = mf[mf['cmdb_id'].isin(['MG01','MG02'])].copy()

# Defensive: ensure numeric fields
mf_mg['duration_points'] = pd.to_numeric(mf_mg.get('duration_points', 0), errors='coerce').fillna(0).astype(int)
mf_mg['severity_score'] = pd.to_numeric(mf_mg.get('severity_score'), errors='coerce')
mf_mg['max_value'] = pd.to_numeric(mf_mg.get('max_value'), errors='coerce')
mf_mg['min_value'] = pd.to_numeric(mf_mg.get('min_value'), errors='coerce')

# Helper: mode function for anomaly_type_majority
def mode_series(s):
    if s.empty:
        return np.nan
    modes = s.mode(dropna=True)
    return modes.iloc[0] if not modes.empty else s.iloc[0]

# Aggregate per (cmdb_id, kpi_name)
agg = mf_mg.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    fault_count = ('kpi_name','count'),
    distinct_faults_duration_points = ('duration_points','sum'),
    earliest_fault_start = ('fault_start_ts','min'),
    anomaly_type_majority = ('anomaly_type', lambda x: mode_series(x.astype(str))),
    max_severity = ('severity_score','max'),
    max_value = ('max_value','max'),
    min_value = ('min_value','min')
)

# Join global p95/p5 thresholds from metric_grouped_full (global thresholds precomputed BEFORE window)
if not mg_full.empty:
    mg_thresh = mg_full[['cmdb_id','kpi_name','p95','p5']].copy()
    agg = agg.merge(mg_thresh, on=['cmdb_id','kpi_name'], how='left')
else:
    agg['p95'] = np.nan
    agg['p5'] = np.nan

# Order by max_severity desc (and fault_count desc as tiebreaker), pick top 10
agg_sorted = agg.sort_values(['max_severity','fault_count'], ascending=[False, False]).reset_index(drop=True)
mg_top10_summary = agg_sorted.head(10)[[
    'cmdb_id','kpi_name','fault_count','distinct_faults_duration_points',
    'earliest_fault_start','anomaly_type_majority','max_severity','p95','p5','max_value','min_value'
]].copy()

# For each selected KPI group, collect up to 5 earliest anomaly point samples from metric_anomaly_points
sample_rows = []
if not mp.empty and not mg_top10_summary.empty:
    for _, row in mg_top10_summary.iterrows():
        cm = row['cmdb_id']
        kp = row['kpi_name']
        samp = mp[(mp['cmdb_id']==cm) & (mp['kpi_name']==kp)].copy()
        if not samp.empty:
            samp = samp.sort_values('ts').head(5)
            # Keep only requested sample columns
            samp_sel = samp[['ts','value','high_anom','low_anom']].copy()
            samp_sel.insert(0, 'kpi_name', kp)
            samp_sel.insert(0, 'cmdb_id', cm)
            sample_rows.append(samp_sel)
# Concatenate samples into one DataFrame for easy inspection; may be empty
if sample_rows:
    mg_kpi_sample_points = pd.concat(sample_rows, ignore_index=True)
else:
    mg_kpi_sample_points = pd.DataFrame(columns=['cmdb_id','kpi_name','ts','value','high_anom','low_anom'])

# Save full aggregated summary to kernel variable mg_metric_faults_summary (already exists) and samples mg_kpi_sample_points
# Return only aggregated top-10 summary and a short pointer about saved samples
sample_info = f"mg_kpi_sample_points saved with {mg_kpi_sample_points.shape[0]} rows (up to 5 samples per KPI group)."

# Display compact results: aggregated rows (top 10) and sample_info string
mg_top10_summary, sample_info

```
Out[9]:
```
Summary — top 10 KPI groups (MG01 / MG02) ranked by max severity:

1. MG01 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps
   - fault_count: 14, duration_points: 14
   - earliest_fault_start: 2021-03-25 06:31:00+08:00
   - anomaly_type_majority: low
   - max_severity: 2.857143
   - p95: 0.029155, p5: 0.000000
   - max_value: 0.083300, min_value: 0.000000

2. MG01 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead
   - fault_count: 14, duration_points: 14
   - earliest: 2021-03-25 06:31:00+08:00
   - majority: low
   - max_severity: 2.857143
   - p95: 0.513345, p5: 0.000000
   - max_value: 1.466700, min_value: 0.000000

3. MG02 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps
   - fault_count: 14, duration_points: 14
   - earliest: 2021-03-25 06:31:00+08:00
   - majority: low
   - max_severity: 2.857143
   - p95: 0.350000, p5: 0.000000
   - max_value: 1.000000, min_value: 0.000000

4. MG02 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps
   - fault_count: 13, duration_points: 13
   - earliest: 2021-03-25 06:30:00+08:00
   - majority: low
   - max_severity: 2.500000
   - p95: 0.400000, p5: 0.000000
   - max_value: 1.000000, min_value: 0.000000

5. MG02 — OSLinux-CPU_CPU-3_SingleCpuidle
   - fault_count: 13, duration_points: 13
   - earliest: 2021-03-25 06:31:00+08:00
   - majority: low
   - max_severity: 2.386341
   - p95: 41.563847, p5: 0.000000
   - max_value: 99.185505, min_value: 0.000000

6. MG01 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps
   - fault_count: 12, duration_points: 12
   - earliest: 2021-03-25 06:30:00+08:00
   - majority: low
   - max_severity: 2.000000
   - p95: 3.500000, p5: 0.000000
   - max_value: 7.000000, min_value: 0.000000

7. MG01 — OSLinux-CPU_CPU_CPUWio
   - fault_count: 6, duration_points: 6
   - earliest: 2021-03-25 06:32:00+08:00
   - majority: low
   - max_severity: 1.954432
   - p95: 0.276397, p5: 0.012500
   - max_value: 0.540200, min_value: 0.012471

8. MG01 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps
   - fault_count: 11, duration_points: 11
   - earliest: 2021-03-25 06:31:00+08:00
   - majority: low
   - max_severity: 1.951220
   - p95: 2.050000, p5: 0.000000
   - max_value: 4.000000, min_value: 0.000000

9. MG01 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps
   - fault_count: 10, duration_points: 10
   - earliest: 2021-03-25 06:31:00+08:00
   - majority: low
   - max_severity: 1.818182
   - p95: 0.550000, p5: 0.000000
   - max_value: 1.000000, min_value: 0.000000

10. MG01 — OSLinux-CPU_CPU-0_SingleCpuUtil
    - fault_count: 2, duration_points: 2
    - earliest: 2021-03-25 06:38:00+08:00
    - majority: high
    - max_severity: 1.783887
    - p95: 53.924935, p5: 0.777915
    - max_value: 96.196000, min_value: 0.747900

Per-point samples:
- Up to 5 earliest anomaly point samples for each of these KPI groups have been saved to kernel variable mg_kpi_sample_points (47 rows total). Use that variable to inspect sample timestamps, values, and the high_anom/low_anom flags.

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id                                         kpi_name  fault_count  distinct_faults_duration_points      earliest_fault_start anomaly_type_majority  max_severity        p95        p5  max_value  min_value
0    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps           14                               14 2021-03-25 06:31:00+08:00                   low      2.857143   0.029155  0.000000   0.083300   0.000000
1    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead           14                               14 2021-03-25 06:31:00+08:00                   low      2.857143   0.513345  0.000000   1.466700   0.000000
2    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps           14                               14 2021-03-25 06:31:00+08:00                   low      2.857143   0.350000  0.000000   1.000000   0.000000
3    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps           13                               13 2021-03-25 06:30:00+08:00                   low      2.500000   0.400000  0.000000   1.000000   0.000000
4    MG02                  OSLinux-CPU_CPU-3_SingleCpuidle           13                               13 2021-03-25 06:31:00+08:00                   low      2.386341  41.563847  0.000000  99.185505   0.000000
5    MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps           12                               12 2021-03-25 06:30:00+08:00                   low      2.000000   3.500000  0.000000   7.000000   0.000000
6    MG01                           OSLinux-CPU_CPU_CPUWio            6                                6 2021-03-25 06:32:00+08:00                   low      1.954432   0.276397  0.012500   0.540200   0.012471
7    MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps           11                               11 2021-03-25 06:31:00+08:00                   low      1.951220   2.050000  0.000000   4.000000   0.000000
8    MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps           10                               10 2021-03-25 06:31:00+08:00                   low      1.818182   0.550000  0.000000   1.000000   0.000000
9    MG01                  OSLinux-CPU_CPU-0_SingleCpuUtil            2                                2 2021-03-25 06:38:00+08:00                  high      1.783887  53.924935  0.777915  96.196000   0.747900, 'mg_kpi_sample_points saved with 47 rows (up to 5 samples per KPI group).')```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing mg_kpi_sample_points and metric_grouped_full from kernel
mg_samples = mg_kpi_sample_points.copy() if 'mg_kpi_sample_points' in globals() else pd.DataFrame()
mg_thresh = metric_grouped_full[['cmdb_id','kpi_name','p95','p5']].copy() if 'metric_grouped_full' in globals() else pd.DataFrame()

# Ensure expected columns exist
for col in ['cmdb_id','kpi_name','ts','value','high_anom','low_anom']:
    if col not in mg_samples.columns:
        mg_samples[col] = pd.NA

# Normalize ts to timezone-aware UTC+8
# Use utc=True to safely interpret naive timestamps as UTC, then convert to Asia/Shanghai (UTC+8)
mg_samples['ts'] = pd.to_datetime(mg_samples['ts'], utc=True).dt.tz_convert(tz)

# Merge thresholds (p95/p5) for severity calculation
mg_samples = mg_samples.merge(mg_thresh, on=['cmdb_id','kpi_name'], how='left')

# Ensure numeric value/p95/p5
mg_samples['value'] = pd.to_numeric(mg_samples['value'], errors='coerce')
mg_samples['p95'] = pd.to_numeric(mg_samples['p95'], errors='coerce')
mg_samples['p5'] = pd.to_numeric(mg_samples['p5'], errors='coerce')

# Compute severity per instruction: for high_anom use value/p95; for low_anom use value/p5.
# If both flags true, prefer high_anom severity. Guard against zero/NaN thresholds.
mg_samples['high_anom'] = mg_samples['high_anom'].astype(bool)
mg_samples['low_anom'] = mg_samples['low_anom'].astype(bool)

mg_samples['severity'] = np.nan
mask_high = mg_samples['high_anom'] & mg_samples['p95'].notna() & (mg_samples['p95'] != 0)
mg_samples.loc[mask_high, 'severity'] = mg_samples.loc[mask_high, 'value'] / mg_samples.loc[mask_high, 'p95']
# low severity only where not already high (to prefer high when both true)
mask_low = (~mask_high) & mg_samples['low_anom'] & mg_samples['p5'].notna() & (mg_samples['p5'] != 0)
mg_samples.loc[mask_low, 'severity'] = mg_samples.loc[mask_low, 'value'] / mg_samples.loc[mask_low, 'p5']

# Sort by ts asc (earliest first), and within same ts by severity desc
mg_samples_sorted = mg_samples.sort_values(by=['ts','severity'], ascending=[True, False]).reset_index(drop=True)

# Select requested columns and limit to 20 rows
mg_kpi_sample_earliest_display = mg_samples_sorted[['cmdb_id','kpi_name','ts','value','high_anom','low_anom','severity']].head(20).copy()

# Save variable in kernel (already assigned) and display the compact table
mg_kpi_sample_earliest_display

```
Out[10]:
```
Summary of MG01/MG02 earliest anomaly samples (timestamps in UTC+8):

- What was returned:
  - Up to 20 earliest anomaly point samples (sorted by timestamp) for MG01 and MG02 were produced and saved to kernel variable mg_kpi_sample_earliest_display.
  - Per-point samples are also stored in mg_kpi_sample_points (47 rows saved earlier) for deeper inspection.

- Key observations from the earliest 20 samples:
  - Time window covered: samples start at 2021-03-25 06:30:00+08:00 and run through ~06:35:00+08:00.
  - Many samples are low_anom (low anomalies) with value = 0.0 — these are for disk and CPU-related KPIs (e.g., DSKBps, DSKTps, SingleCpuidle). Severity is NaN for those because the global p5 thresholds are zero (division by zero avoided).
  - Notable high_anom rows (non-zero severity) at early times:
    - MG01 / sda_DSKRTps at 06:31 — value 0.0833, high_anom True, severity ≈ 2.857
    - MG01 / sda_DSKRead at 06:31 — value 1.4667, high_anom True, severity ≈ 2.857
    - MG01 / CPU_CPU_CPUWio at 06:32 — value 0.5402, high_anom True, severity ≈ 1.954
  - Many other disk-related metrics show repeated zero readings flagged as low_anom at multiple timestamps (e.g., sda/sdb DSKBps, DSKTps, and CPU single-core idle).

- Interpretation / likely issues:
  - The data indicates disk I/O and filesystem anomalies on MG01 and MG02 during the incident window:
    - Several disk throughput metrics are zero (low_anom), suggesting loss of expected I/O activity or measurement dropouts.
    - There are concurrent high anomalies in disk read/RT and CPU wait (CPU WIO) on MG01 shortly after 06:31, indicating actual I/O latency or backlog for that host.
  - These patterns point to local disk/subsystem performance problems on MG01/MG02 (investigate disks sda/sdb, disk controllers, or underlying storage).

- Next steps you can request:
  - Show the full mg_kpi_sample_points (47 rows) to inspect per-point values and flags.
  - Correlate these samples with trace_anomaly_pairs involving MG01/MG02 to see caller impact.
  - Inspect system logs or storage device metrics around 06:30–06:35 on MG01/MG02.

If you want, I can now (a) display the saved mg_kpi_sample_points, (b) filter samples by KPI name, or (c) correlate these samples with trace anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                         kpi_name                        ts   value  high_anom  low_anom  severity
0     MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps 2021-03-25 06:30:00+08:00  0.0000      False      True       NaN
1     MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps 2021-03-25 06:30:00+08:00  0.0000      False      True       NaN
2     MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-25 06:31:00+08:00  0.0833       True     False  2.857143
3     MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-25 06:31:00+08:00  1.4667       True     False  2.857143
4     MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps 2021-03-25 06:31:00+08:00  0.0000      False      True       NaN
5     MG02                  OSLinux-CPU_CPU-3_SingleCpuidle 2021-03-25 06:31:00+08:00  0.0000      False      True       NaN
6     MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps 2021-03-25 06:31:00+08:00  0.0000      False      True       NaN
7     MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps 2021-03-25 06:31:00+08:00  0.0000      False      True       NaN
8     MG01                           OSLinux-CPU_CPU_CPUWio 2021-03-25 06:32:00+08:00  0.5402       True     False  1.954432
9     MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps 2021-03-25 06:32:00+08:00  0.0000      False      True       NaN
10    MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps 2021-03-25 06:32:00+08:00  0.0000      False      True       NaN
11    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-25 06:33:00+08:00  0.0000      False      True       NaN
12    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-25 06:33:00+08:00  0.0000      False      True       NaN
13    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps 2021-03-25 06:33:00+08:00  0.0000      False      True       NaN
14    MG02                  OSLinux-CPU_CPU-3_SingleCpuidle 2021-03-25 06:33:00+08:00  0.0000      False      True       NaN
15    MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps 2021-03-25 06:33:00+08:00  0.0000      False      True       NaN
16    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps 2021-03-25 06:34:00+08:00  0.0000      False      True       NaN
17    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-25 06:35:00+08:00  0.0000      False      True       NaN
18    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-25 06:35:00+08:00  0.0000      False      True       NaN
19    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps 2021-03-25 06:35:00+08:00  0.0000      False      True       NaN```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing mg_kpi_sample_points (per-point samples) from kernel
mg_samples = mg_kpi_sample_points.copy() if 'mg_kpi_sample_points' in globals() else pd.DataFrame()

# Ensure required columns exist
for c in ['cmdb_id','kpi_name','ts','value','high_anom','low_anom','p95','p5']:
    if c not in mg_samples.columns:
        mg_samples[c] = pd.NA

# Normalize ts to timezone-aware UTC+8
mg_samples['ts'] = pd.to_datetime(mg_samples['ts'], utc=True).dt.tz_convert(tz)

# Ensure numeric columns
mg_samples['value'] = pd.to_numeric(mg_samples['value'], errors='coerce')
mg_samples['p95'] = pd.to_numeric(mg_samples['p95'], errors='coerce')
mg_samples['p5'] = pd.to_numeric(mg_samples['p5'], errors='coerce')

# Ensure boolean flags
mg_samples['high_anom'] = mg_samples['high_anom'].astype(bool)
mg_samples['low_anom'] = mg_samples['low_anom'].astype(bool)

# Compute severity per instruction (prefer high_anom when both true)
mg_samples['severity'] = np.nan
mask_high = mg_samples['high_anom'] & mg_samples['p95'].notna() & (mg_samples['p95'] != 0)
mg_samples.loc[mask_high, 'severity'] = mg_samples.loc[mask_high, 'value'] / mg_samples.loc[mask_high, 'p95']
# low only where high not applied
mask_low = (~mask_high) & mg_samples['low_anom'] & mg_samples['p5'].notna() & (mg_samples['p5'] != 0)
mg_samples.loc[mask_low, 'severity'] = mg_samples.loc[mask_low, 'value'] / mg_samples.loc[mask_low, 'p5']

# Sort by severity desc (NaNs go last) then ts asc
mg_samples_sorted = mg_samples.copy()
# Replace NaN severity with -inf for sorting descending while preserving NaNs at end after filtering
mg_samples_sorted['severity_sort'] = mg_samples_sorted['severity'].replace({np.nan: -np.inf})
mg_samples_sorted = mg_samples_sorted.sort_values(by=['severity_sort','ts'], ascending=[False, True]).drop(columns=['severity_sort'])

# Select requested columns and top 10
mg_kpi_sample_top_by_severity = mg_samples_sorted[['cmdb_id','kpi_name','ts','value','high_anom','low_anom','p95','p5','severity']].head(10).reset_index(drop=True)

# Display compact result (variable output)
mg_kpi_sample_top_by_severity

```
Out[11]:
```
```python
# Summarize the top-10 MG01/MG02 anomaly samples by severity (plain English)
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
df = mg_kpi_sample_top_by_severity if 'mg_kpi_sample_top_by_severity' in globals() else pd.DataFrame()

if df.empty:
    summary = "No sample anomaly points available in mg_kpi_sample_top_by_severity."
else:
    # Basic counts and observations
    total_rows = df.shape[0]
    nan_sev = int(df['severity'].isna().sum())
    high_count = int(df['high_anom'].sum())
    low_count = int(df['low_anom'].sum())
    earliest = pd.to_datetime(df['ts']).min().tz_convert(tz)
    latest = pd.to_datetime(df['ts']).max().tz_convert(tz)
    # Example notable rows (first 3)
    examples = []
    for r in df.head(3).itertuples(index=False):
        examples.append(f"{r.cmdb_id}/{r.kpi_name} at {pd.to_datetime(r.ts).tz_convert(tz)} value={r.value} high={r.high_anom} low={r.low_anom} sev={r.severity}")
    examples_s = "; ".join(examples)

    summary = (
        f"Top-10 MG01/MG02 anomaly samples (saved to mg_kpi_sample_top_by_severity):\n\n"
        f"- Rows returned: {total_rows}. Earliest sample: {earliest}, latest sample: {latest} (UTC+8).\n"
        f"- Severity values: {nan_sev} of {total_rows} are NaN (could not be computed because p95/p5 thresholds are missing or zero).\n"
        f"- Flags: {high_count} high_anom rows and {low_count} low_anom rows in these top-10 samples.\n\n"
        f"- Observations from the top examples: {examples_s}.\n"
        f"- Most samples are low_anom with value = 0.0 (disk I/O and CPU idle metrics), producing NaN severity when p5 is zero or missing.\n"
        f"- A few high_anom samples have non-zero values (e.g., disk read/RT) but severity is NaN here because the matching p95 was not present in these saved samples.\n\n"
        f"Next steps:\n"
        f"1) Recompute severity by joining mg_kpi_sample_points with metric_grouped_full (to obtain p95/p5) and re-running the severity formula.\n"
        f"2) Inspect mg_kpi_sample_points (47 rows saved earlier) or mg_metric_faults_summary for full KPI context and thresholds.\n"
        f"\n"
        f"If you want, I can recompute severity now by merging the samples with metric_grouped_full and return the top points by true severity."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                         kpi_name                        ts   value  high_anom  low_anom  p95  p5  severity
0    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps 2021-03-25 06:30:00+08:00  0.0000      False      True  NaN NaN       NaN
1    MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps 2021-03-25 06:30:00+08:00  0.0000      False      True  NaN NaN       NaN
2    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-25 06:31:00+08:00  0.0833       True     False  NaN NaN       NaN
3    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-25 06:31:00+08:00  1.4667       True     False  NaN NaN       NaN
4    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps 2021-03-25 06:31:00+08:00  0.0000      False      True  NaN NaN       NaN
5    MG02                  OSLinux-CPU_CPU-3_SingleCpuidle 2021-03-25 06:31:00+08:00  0.0000      False      True  NaN NaN       NaN
6    MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps 2021-03-25 06:31:00+08:00  0.0000      False      True  NaN NaN       NaN
7    MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps 2021-03-25 06:31:00+08:00  0.0000      False      True  NaN NaN       NaN
8    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps 2021-03-25 06:32:00+08:00  0.0000      False      True  NaN NaN       NaN
9    MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps 2021-03-25 06:32:00+08:00  0.0000      False      True  NaN NaN       NaN```
```